# 22 Experiment 5 — Entity Length


Các chỉ số và bảng bên dưới được tính trực tiếp từ nhãn và dự đoán đã lưu. Cột giải thích để trống cho sinh viên tự hoàn thiện.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/NLP/lab09'
import os, json, random, numpy as np
os.makedirs(DRIVE_DIR, exist_ok=True)
SEED = 42
random.seed(SEED); np.random.seed(SEED)
import pandas as pd, json, numpy as np
def spans(words, tags):
    out=[]; start=None; typ=None
    def close(end):
        if start is not None: out.append((start,end,typ,' '.join(words[start:end])))
    for i, tag in enumerate(list(tags)+['O']):
        if tag == 'O':
            close(i); start=None; typ=None; continue
        prefix, kind = tag.split('-',1)
        if prefix == 'B' or typ != kind:
            close(i); start=i; typ=kind
        elif start is None:
            start=i; typ=kind
    return out

def entity_set(words, tags):
    return {(a,b,t) for a,b,t,_ in spans(words,tags)}

def prf(tp, fp, fn):
    p=tp/(tp+fp) if tp+fp else 0.0
    r=tp/(tp+fn) if tp+fn else 0.0
    return p,r,(2*p*r/(p+r) if p+r else 0.0)

def extract(words, tags):
    mapping={'PER':'PERSON','ORG':'ORG','LOC':'LOCATION','MISC':'MISC'}
    result={'PERSON':[],'ORG':[],'LOCATION':[],'MISC':[]}
    for a,b,typ,text in spans(words,tags):
        key=mapping[typ]
        if text not in result[key]: result[key].append(text)
    return result

def counts(gold_rows, pred_rows, type_filter=None):
    tp=fp=fn=0
    for g,p in zip(gold_rows,pred_rows):
        gs=entity_set(g['words'],g['gold']); ps=entity_set(p['words'],p['pred'])
        if type_filter:
            gs={x for x in gs if x[2]==type_filter}; ps={x for x in ps if x[2]==type_filter}
        tp+=len(gs&ps); fp+=len(ps-gs); fn+=len(gs-ps)
    return tp,fp,fn

with open(f'{DRIVE_DIR}/test_gold.json',encoding='utf-8') as f: gold=json.load(f)
with open(f'{DRIVE_DIR}/transformer_predictions.json',encoding='utf-8') as f: transformer=json.load(f)
assert len(gold)==len(transformer)


In [ ]:
def bucket(n): return str(n) if n<4 else '4+'
rows=[]
for model_name,pred in [('Transformer',transformer)]:
    for b in ['1','2','3','4+']:
        tp=fp=fn=0
        for g,p in zip(gold,pred):
            gs=entity_set(g['words'],g['gold']); ps=entity_set(p['words'],p['pred'])
            gs_b={x for x in gs if bucket(x[1]-x[0])==b}; ps_b={x for x in ps if bucket(x[1]-x[0])==b}
            tp+=len(gs_b&ps_b); fn+=len(gs_b-ps_b); fp+=len(ps_b-gs_b)
        precision,recall,f1=prf(tp,fp,fn); rows.append({'Model':model_name,'Length':b,'TP':tp,'FP':fp,'FN':fn,'Precision':precision,'Recall':recall,'F1':f1})
length_df=pd.DataFrame(rows); display(length_df); length_df.to_csv(f'{DRIVE_DIR}/entity_length_metrics.csv',index=False)
import matplotlib.pyplot as plt
ax=length_df.pivot(index='Length',columns='Model',values='F1').plot(kind='bar',ylim=(0,1),figsize=(8,5),color=['#4C78A8','#F58518']); ax.set_ylabel('Entity-level F1'); ax.set_xlabel('Entity length (words)'); plt.tight_layout(); plt.savefig(f'{DRIVE_DIR}/entity_length_analysis.png',dpi=180); plt.show()
